In [5]:
import warnings
warnings.filterwarnings('ignore')

import evaluate
import pandas as pd
import re
import json
import mlflow
from datasets import load_dataset, Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, pipeline

## Data Preparation

In [6]:
# Loading the dataset
df = load_dataset("Tobi-Bueck/customer-support-tickets")['train'].to_pandas()

# Cleaning the text data
def clean_text(text):
    if pd.isna(text):
        return ""

    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()  # Convert all text to lowercase

    return text

df['cleaned_body'] = df['body'].apply(clean_text)
df['cleaned_subject'] = df['subject'].apply(clean_text)

# Combining subject and body for input

def combine_subject_body(row):
    subject = row['cleaned_subject']
    body = row['cleaned_body']
    if subject:
        return subject + '. ' + body
    else:
        return body


df['subject_body'] = df.apply(combine_subject_body, axis=1)


In [7]:
df['subject_body']

0        wesentlicher sicherheitsvorfall. sehr geehrtes...
1        account disruption. dear customer support team...
2        query about smart home system integration feat...
3        inquiry regarding invoice details. dear custom...
4        question about marketing agency software compa...
                               ...                        
61760    assistance needed for ifttt docker integration...
61761    bitten um unterstützung bei der integration. s...
61762    hello customer support, i am inquiring about t...
61763    hilfe bei digitalen strategie-problemen. die q...
61764    optimierung ihrer datenanalyse-plattform erlei...
Name: subject_body, Length: 61765, dtype: object